In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Trading_sentiment_platform").getOrCreate()

In [0]:
# Install NLP + transformer libraries
%pip install transformers torch spacy vaderSentiment hf_transfer


In [0]:
%restart_python

In [0]:
import transformers
import spacy
import torch
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [0]:
# reference catalog and schema usage
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA sec_filings")

In [0]:
# load sentiment data into a dataframe
sentiment_df = spark.read.table("stg_cleaned_filings")\
                .select("cik", "company_name", "filing_date", "accessionNumber", "mda_text")

display(sentiment_df)
sentiment_df.count()

In [0]:
# !python -m spacy download en_core_web_sm

In [0]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = ("ProsusAI/finbert")
tokenizer = AutoTokenizer.from_pretrained(model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name)
sentiment_model.eval()

In [0]:
# Define sentiment-scoring function 
def get_sentiment(text):
    if text is None or text.strip() == "":
        return (0.0, "neutral", 0.0, 0.0, 0.0)
    
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
    
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    labels = ["positive", "neutral", "negative"]
    
    sentiment_label = labels[int(torch.argmax(probs))]
    
    # Compound score = P - N
    compound = float(probs[0] - probs[2])
    
    return (
        compound,
        sentiment_label,
        float(probs[0]),
        float(probs[1]),
        float(probs[2])
    )


In [0]:
# transform sentiment_df into pandas for finBERT
sentiment_pd = sentiment_df.toPandas()
print(sentiment_pd)

In [0]:
# Apply sentiment scoring
results = []

for idx, row in sentiment_pd.iterrows():
    score, label, pos, neu, neg = get_sentiment(row["mda_text"])
    
    results.append({
        "cik": row["cik"],
        "company_name": row["company_name"],
        "filing_date": row["filing_date"],
        "accession_number": row["accessionNumber"],
        "sentiment_score": score,
        "sentiment_label": label,
        "positive_prob": pos,
        "neutral_prob": neu,
        "negative_prob": neg
    })


In [0]:
# Convert back to spark
sentiment_df = spark.createDataFrame(results)
display(sentiment_df.limit(10))

In [0]:
from pyspark.sql import functions as F
# Create a schema for the sentiment dataframe to store in a delta table
spark.sql("""CREATE OR REPLACE TABLE sec_filings.fact_sentiment(
    accession_number STRING,
    cik STRING,
    company_name STRING,
    filing_date DATE,
    negative_prob DOUBLE,
    neutral_prob DOUBLE,
    positive_prob DOUBLE,
    sentiment_label STRING,
    sentiment_score DOUBLE,
    processed_timestamp TIMESTAMP
)
USING DELTA;""")

# Add timestamp to sentiment dataframe
sentiment_df = sentiment_df.withColumn("processed_timestamp", F.current_timestamp())
display(sentiment_df.limit(10))

In [0]:
# Add sentiment_df data into fact_sentiment delta table
sentiment_df.write.format("delta").mode("append").saveAsTable("sec_filings.fact_sentiment")